# 🚀 Causal Multimodal Reasoning for Crisis Generalization

Notebook này được thiết lập sẵn môi trường chuẩn khoa học để huấn luyện mô hình **Causal Crisis GNN** trên tập dữ liệu CrisisMMD v2.0.

## Bước 1: Tải Tập Dữ Liệu (CrisisMMD v2.0)
Thay vì phải tải lên Google Drive, chúng ta sẽ dùng `wget` để tải trực tiếp và giải nén dataset vào môi trường Colab (Nhanh và độc lập).

In [ ]:
import os

if not os.path.exists("/content/CrisisMMD_v2.0"):
    print("\u2b07️ Đang tải xuống CrisisMMD v2.0 dataset (Khoảng 1.8GB)...")
    !wget https://crisisnlp.qcri.org/data/crisismmd/CrisisMMD_v2.0.tar.gz -O /content/CrisisMMD_v2.0.tar.gz
    print("\ud83d\udce6 Đang giải nén dữ liệu...")
    !tar -xzf /content/CrisisMMD_v2.0.tar.gz -C /content/
    !rm /content/CrisisMMD_v2.0.tar.gz # Xóa file nén để tiết kiệm ổ cứng
    print("\u2705 Tải và giải nén dataset thành công!")
else:
    print("\u2705 Dataset đã tồn tại! Có thể bỏ qua bước này.")

## Bước 2: Kéo Mã Nguồn Khoa Học từ Github
Tải trực tiếp Repository hoàn chỉnh (*shallow clone* để giảm thiểu unnecesary size) cùng toàn bộ thuật toán chúng ta đã xây dựng.

In [ ]:
import os
import sys

if not os.path.exists("/content/CrisisSummarization-CausalGNN"):
    !git clone --depth 1 https://github.com/Thanh-000/CrisisSummarization-CausalGNN.git

%cd /content/CrisisSummarization-CausalGNN

# Báo cho Python biết vị trí kho code
REPO_DIR = "/content/CrisisSummarization-CausalGNN"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Cài đặt các thư viện lõi
!pip install -q -r requirements.txt
!pip install -q open_clip_torch
!pip install -q faiss-gpu faiss-cpu

## Bước 3: Khởi chạy Ablation Mode (Huấn Luyện và Đánh Giá)
Mặc định đường dẫn Dataset là `/content/CrisisMMD_v2.0` - đã được tải về ở Mục 1.

In [ ]:
from src.trainers.causal_crisis_trainer import run_ablation_suite

run_ablation_suite(
    dataset_path='/content/CrisisMMD_v2.0',  # <-- Tự động trỏ vào thư mục dữ liệu vừa tải
    seeds=[42],           # Số ngẫu nhiên (chỉ test 1 lần để chạy nhanh)
    tasks=["task1"],      # Phân loại độ Hữu Ích (Informative) của thảm họa
    few_shot_sizes=[50],  # Test chế độ 50 mẫu gán nhãn
    device="cuda",        # Autodetect GPU Colab
    results_csv="./results/ablation_test_results.csv"
)

## Bước 4: Kiểm tra kết quả Xuất Ra

In [ ]:
import pandas as pd

try:
    df_ablation = pd.read_csv("./results/ablation_test_results.csv")
    display(df_ablation.tail(7)) # Sẽ hiển thị điểm F1 Score, Accuracy... của 7 Variants Ablation/Intervention
except Exception as e:
    print("Chưa có file kết quả hoặc có lỗi:", e)